In [1]:
   import json
   import os
   import pandas as pd

In [2]:
   ordner = "../data/toxic_and_baseline_random_oss"
   dateien = os.listdir(ordner)
   print("Anzahl Gespräche:", len(dateien))
   print(dateien[:5])

Anzahl Gespräche: 18684
['.ipynb_checkpoints', 'discussion_1000_round_0_for_topic_15_con_toxic_level_moderate.json', 'discussion_1000_round_0_for_topic_15_neither_toxic_level_no.json', 'discussion_1000_round_0_for_topic_15_pro_toxic_level_mild.json', 'discussion_1000_round_1_for_topic_15_con_toxic_level_moderate.json']


In [ ]:
   with open(os.path.join(ordner, dateien[0]), encoding="utf-8") as f:
       gespraech = json.load(f)

   print("Thema:", gespraech["proposition"])
   print("Anzahl Runden:", gespraech["nrounds"])
   print("Einigung erreicht:", gespraech["in_alignment"])
   

In [ ]:
   for seite, agenten in gespraech["active_agents"].items():
       for a in agenten:
           print(seite.upper(), "->", a["description"])

In [ ]:
   for s in gespraech["opening_statements"]:
       print(s["agent"], ":")
       print(s["opening_statement"][:500], "...\n")

In [ ]:
   for runde, antworten in gespraech["discussion_history"].items():
       if runde == "opening_statements":
           continue
       print("=====", runde, "=====")
       for a in antworten:
           print(a["agent"], ":")
           print(a["next_answer"][:500], "...\n")

In [ ]:
   zeilen = []
   for name in dateien:
       with open(os.path.join(ordner, name), encoding="utf-8") as f:
           g = json.load(f)
       toxizitaet = name.split("toxic_level_")[1].replace(".json", "")
       zeilen.append({
           "datei": name,
           "thema": g["proposition"],
           "toxizitaet": toxizitaet,
           "runden": g["nrounds"],
           "einigung": g["in_alignment"],
       })

   df = pd.DataFrame(zeilen)
   df.head()

In [ ]:
   df.groupby("toxizitaet").agg(
       anzahl=("datei", "count"),
       runden_durchschnitt=("runden", "mean"),
       einigungsquote=("einigung", "mean"),
   ).round(2)

   ## Beobachtungen (17.10.2026)

   - **Runden:** Ohne Toxizität dauert eine Debatte im Schnitt 2,1 Runden, bei heavy 4,3 Runden → toxische Debatten dauern länger (bestätigt MAD-Paper).
   - **Einigungsquote:** Ohne Toxizität 35 % Einigung, bei heavy nur 16 % → Toxizität erschwert Einigung.
   - **Offene Frage an Prof. Mangold:** Warum enden so viele Debatten ohne Einigung? Wichtig für H2 (Gewinner nur bei Einigung messbar).
   - **Personas:** Manche Personas haben schon Namen (z. B. „Jamie Reed“) → können Geschlecht andeuten, beim Gender-Parameter beachten.
   - **Toxische Sprache:** Sarkasmus, Lächerlich machen, Herablassung → mögliche Kategorien für H3.

In [3]:
   test = sorted([d for d in dateien if d.startswith("discussion_992_")])
   print("Dateien für discussion_992:", len(test))
   for name in test:
       with open(os.path.join(ordner, name), encoding="utf-8") as f:
           g = json.load(f)
       print(name, "| Runden:", g["nrounds"], "| Einigung:", g["in_alignment"])

Dateien für discussion_992: 22
discussion_992_round_0_for_topic_18_con_toxic_level_heavy.json | Runden: 1 | Einigung: False
discussion_992_round_0_for_topic_18_con_toxic_level_moderate.json | Runden: 1 | Einigung: False
discussion_992_round_0_for_topic_18_neither_toxic_level_no.json | Runden: 1 | Einigung: False
discussion_992_round_0_for_topic_18_pro_toxic_level_mild.json | Runden: 1 | Einigung: False
discussion_992_round_10_for_topic_18_con_toxic_level_moderate.json | Runden: 7 | Einigung: True
discussion_992_round_10_for_topic_18_pro_toxic_level_mild.json | Runden: 6 | Einigung: True
discussion_992_round_19_for_topic_18_con_toxic_level_heavy.json | Runden: 4 | Einigung: False
discussion_992_round_1_for_topic_18_con_toxic_level_moderate.json | Runden: 2 | Einigung: False
discussion_992_round_1_for_topic_18_neither_toxic_level_no.json | Runden: 2 | Einigung: False
discussion_992_round_1_for_topic_18_pro_toxic_level_mild.json | Runden: 2 | Einigung: False
discussion_992_round_23_for_to

In [5]:
   import re

   dateien = [d for d in os.listdir(ordner) if d.endswith(".json")]

   zeilen = []
   for name in dateien:
       m = re.match(r"discussion_(\d+)_round_(\d+)_for_topic_(\d+)_(\w+?)_toxic_level_(\w+)\.json", name)
       if m is None:
           continue
       with open(os.path.join(ordner, name), encoding="utf-8") as f:
           g = json.load(f)
       zeilen.append({
           "datei": name,
           "diskussion": int(m[1]),
           "thema_nr": int(m[3]),
           "toxizitaet": m[5],
           "runden": g["nrounds"],
           "einigung": g["in_alignment"],
       })

   alle = pd.DataFrame(zeilen)
   debatten = alle.sort_values("runden").groupby(["diskussion", "thema_nr", "toxizitaet"]).tail(1)

   print("Dateien insgesamt:", len(alle))
   print("Echte Debatten (nur Endergebnis):", len(debatten))

Dateien insgesamt: 18683
Echte Debatten (nur Endergebnis): 3998


In [6]:
   debatten.groupby("toxizitaet").agg(
       anzahl=("datei", "count"),
       runden_durchschnitt=("runden", "mean"),
       einigungsquote=("einigung", "mean"),
   ).round(2)

,anzahl,runden_durchschnitt,einigungsquote
toxizitaet,,,
heavy,998,5.83,0.99
mild,1000,4.57,1.00
moderate,1000,5.36,1.00
no,1000,2.93,1.00


## Beobachtungen (aktualisiert 19.10.2026)

**Datensatz:** Vollständige GPT-OSS-Daten von Frau Moldovan-Mauer (FAUbox): 18.683 Dateien = 3.998 echte Debatten.
Pro Debatte gibt es mehrere Dateien (Zwischenstand nach jeder Runde) → nur die Datei mit den meisten Runden verwenden.

**Ergebnisse (nur Endergebnisse):**
- no: 2,93 Runden | mild: 4,57 | moderate: 5,36 | heavy: 5,83
- Einigung in fast 100 % der Debatten → H2 (Gewinner) gut auswertbar

**Offene Fragen:**
- Was zählt die Zahl hinter "round_" im Dateinamen?
- Warum ist der Effekt hier größer als die ca. 25 % aus dem Paper? (im Paper nachlesen)

**Weitere Beobachtungen:**
- Manche Personas haben Namen (z. B. "Jamie Reed") → beim Gender-Parameter beachten
- Toxische Sprache: Sarkasmus, Lächerlich machen, Herablassung → mögliche Kategorien für H3